# Customer Churn Prediction — Modeling

## Objective

The goal of this notebook is to train and evaluate machine learning models for customer churn prediction.

The business objective is to identify customers who are likely to churn so the company can prioritize retention actions. Since missing a real churner can lead to lost revenue, recall is the primary metric. However, precision is also monitored because false positives can create unnecessary retention costs.

This notebook compares Logistic Regression, Random Forest, and XGBoost, then performs threshold analysis on the best-performing model.

In [1]:
# Customer Churn Prediction - Modeling

import numpy as np
import pandas as pd

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

In [2]:
X_train = np.load("../models/X_train_processed.npy")
X_test = np.load("../models/X_test_processed.npy")
y_train = np.load("../models/y_train.npy")
y_test = np.load("../models/y_test.npy")

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)

X_train shape: (5634, 45)
X_test shape: (1409, 45)
y_train shape: (5634,)
y_test shape: (1409,)


In [3]:
# Train baseline Logistic Regression model

log_reg = LogisticRegression(
    class_weight="balanced",
    max_iter=1000,
    random_state=42
)

log_reg.fit(X_train, y_train);
print("Logistic Regression model trained successfully.")

Logistic Regression model trained successfully.


In [4]:
# Make predictions

y_pred_log_reg = log_reg.predict(X_test)
y_proba_log_reg = log_reg.predict_proba(X_test)[:, 1]

In [5]:
# Evaluate Logistic Regression baseline

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred_log_reg))

print("\nClassification Report:")
print(classification_report(y_test, y_pred_log_reg))

print("Accuracy:", accuracy_score(y_test, y_pred_log_reg))
print("Precision:", precision_score(y_test, y_pred_log_reg))
print("Recall:", recall_score(y_test, y_pred_log_reg))
print("F1-score:", f1_score(y_test, y_pred_log_reg))
print("ROC-AUC:", roc_auc_score(y_test, y_proba_log_reg))

Confusion Matrix:
[[747 288]
 [ 81 293]]

Classification Report:
              precision    recall  f1-score   support

           0       0.90      0.72      0.80      1035
           1       0.50      0.78      0.61       374

    accuracy                           0.74      1409
   macro avg       0.70      0.75      0.71      1409
weighted avg       0.80      0.74      0.75      1409

Accuracy: 0.7381121362668559
Precision: 0.504302925989673
Recall: 0.7834224598930482
F1-score: 0.6136125654450262
ROC-AUC: 0.8416337285902502


### Logistic Regression Baseline Interpretation

The Logistic Regression baseline caught 293 out of 374 churners, giving a recall of 0.7834. This is useful for the business objective because the main goal is to identify customers likely to leave.

The model missed 81 churners, which means some customers who actually left would not be targeted by a retention strategy.

Precision was 0.5043, meaning that around half of the customers flagged as churn risks actually churned. This shows that the model produces many false positives. In business terms, this could lead to unnecessary retention offers being sent to loyal customers.

The ROC-AUC score was 0.8416, which suggests that the model has a good ability to separate churners from non-churners across different thresholds.

Overall, Logistic Regression is a useful baseline because it achieves strong recall. However, it should not be selected as the final model yet because we still need to compare it with Random Forest and XGBoost.

In [6]:
from sklearn.ensemble import RandomForestClassifier

# Train Random Forest model

rf_model = RandomForestClassifier(
    n_estimators=200,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

rf_model.fit(X_train, y_train);

print("Random Forest model trained successfully.")

Random Forest model trained successfully.


In [7]:
# Make predictions

y_pred_rf = rf_model.predict(X_test)
y_proba_rf = rf_model.predict_proba(X_test)[:, 1]

In [8]:
# Evaluate Random Forest

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred_rf))

print("\nClassification Report:")
print(classification_report(y_test, y_pred_rf))

print("Accuracy:", accuracy_score(y_test, y_pred_rf))
print("Precision:", precision_score(y_test, y_pred_rf))
print("Recall:", recall_score(y_test, y_pred_rf))
print("F1-score:", f1_score(y_test, y_pred_rf))
print("ROC-AUC:", roc_auc_score(y_test, y_proba_rf))

Confusion Matrix:
[[921 114]
 [198 176]]

Classification Report:
              precision    recall  f1-score   support

           0       0.82      0.89      0.86      1035
           1       0.61      0.47      0.53       374

    accuracy                           0.78      1409
   macro avg       0.71      0.68      0.69      1409
weighted avg       0.77      0.78      0.77      1409

Accuracy: 0.7785663591199432
Precision: 0.6068965517241379
Recall: 0.47058823529411764
F1-score: 0.5301204819277109
ROC-AUC: 0.8186519930765455


### Random Forest Interpretation

Random Forest achieved higher accuracy and precision than Logistic Regression, but its recall was much lower.

Random Forest caught only 176 out of 374 churners, giving a recall of 0.4706. This means it missed 198 churners, which is a major weakness for a churn prediction project.

Although precision improved to 0.6069, the model sacrificed too much recall. Since the business objective is to identify as many churners as possible, this tradeoff is not acceptable at this stage.

The higher accuracy is misleading because the dataset contains more non-churners than churners. Random Forest performs well on the majority class, but it misses many customers who actually churned.

Compared with Random Forest, Logistic Regression remains the stronger model so far because it has much higher recall, better F1-score, and better ROC-AUC.

In [9]:
from xgboost import XGBClassifier

In [10]:
# Calculate scale_pos_weight using only the training target

negative_count = np.sum(y_train == 0)
positive_count = np.sum(y_train == 1)

scale_pos_weight = negative_count / positive_count

print("Negative class count:", negative_count)
print("Positive class count:", positive_count)
print("scale_pos_weight:", scale_pos_weight)

Negative class count: 4139
Positive class count: 1495
scale_pos_weight: 2.768561872909699


In [11]:
# Train XGBoost model

xgb_model = XGBClassifier(
    n_estimators=200,
    learning_rate=0.05,
    max_depth=3,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=scale_pos_weight,
    eval_metric="logloss",
    random_state=42,
    n_jobs=-1
)

xgb_model.fit(X_train, y_train);
print("XGBoost model trained successfully.")

XGBoost model trained successfully.


In [12]:
# Make predictions

y_pred_xgb = xgb_model.predict(X_test)
y_proba_xgb = xgb_model.predict_proba(X_test)[:, 1]

In [13]:
# Evaluate XGBoost

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred_xgb))

print("\nClassification Report:")
print(classification_report(y_test, y_pred_xgb))

print("Accuracy:", accuracy_score(y_test, y_pred_xgb))
print("Precision:", precision_score(y_test, y_pred_xgb))
print("Recall:", recall_score(y_test, y_pred_xgb))
print("F1-score:", f1_score(y_test, y_pred_xgb))
print("ROC-AUC:", roc_auc_score(y_test, y_proba_xgb))

Confusion Matrix:
[[754 281]
 [ 76 298]]

Classification Report:
              precision    recall  f1-score   support

           0       0.91      0.73      0.81      1035
           1       0.51      0.80      0.63       374

    accuracy                           0.75      1409
   macro avg       0.71      0.76      0.72      1409
weighted avg       0.80      0.75      0.76      1409

Accuracy: 0.7466288147622427
Precision: 0.5146804835924007
Recall: 0.7967914438502673
F1-score: 0.6253934942287513
ROC-AUC: 0.8471944509028908


In [14]:
model_comparison = pd.DataFrame({
    "model": ["Logistic Regression", "Random Forest", "XGBoost"],
    "accuracy": [
        accuracy_score(y_test, y_pred_log_reg),
        accuracy_score(y_test, y_pred_rf),
        accuracy_score(y_test, y_pred_xgb)
    ],
    "precision": [
        precision_score(y_test, y_pred_log_reg),
        precision_score(y_test, y_pred_rf),
        precision_score(y_test, y_pred_xgb)
    ],
    "recall": [
        recall_score(y_test, y_pred_log_reg),
        recall_score(y_test, y_pred_rf),
        recall_score(y_test, y_pred_xgb)
    ],
    "f1_score": [
        f1_score(y_test, y_pred_log_reg),
        f1_score(y_test, y_pred_rf),
        f1_score(y_test, y_pred_xgb)
    ],
    "roc_auc": [
        roc_auc_score(y_test, y_proba_log_reg),
        roc_auc_score(y_test, y_proba_rf),
        roc_auc_score(y_test, y_proba_xgb)
    ]
})

model_comparison

,model,accuracy,precision,recall,f1_score,roc_auc
0,Logistic Regression,0.738112,0.504303,0.783422,0.613613,0.841634
1,Random Forest,0.778566,0.606897,0.470588,0.530120,0.818652
2,XGBoost,0.746629,0.514680,0.796791,0.625393,0.847194


### Model Comparison Interpretation

XGBoost is the strongest model so far. It achieved the highest recall, F1-score, and ROC-AUC among the three models.

Compared with Logistic Regression, XGBoost slightly improved recall from 0.7834 to 0.7968, meaning it caught 298 churners instead of 293. It also improved precision from 0.5043 to 0.5147 and F1-score from 0.6136 to 0.6254.

However, the improvement is modest. XGBoost still produces many false positives, with 281 non-churn customers incorrectly flagged as churn risks. This means retention campaigns based on this model could still create unnecessary costs.

Random Forest had the best precision, but its recall was too low for this project. It missed 198 churners, which makes it less suitable for a recall-focused churn prediction problem.

Based on the current results, XGBoost is the best model so far, but threshold tuning is needed to explore a better recall/precision tradeoff.

## Threshold Analysis

The default classification threshold is 0.50, meaning customers with predicted churn probability greater than or equal to 0.50 are classified as churn.

Because this project prioritizes recall but still needs to control false positives, different thresholds are tested to evaluate the tradeoff between precision, recall, false positives, and false negatives.

In [15]:
# Evaluate XGBoost at different classification thresholds

thresholds = [0.30, 0.35, 0.40, 0.45, 0.50, 0.55, 0.60]

threshold_results = []

for threshold in thresholds:
    y_pred_threshold = (y_proba_xgb >= threshold).astype(int)
    
    threshold_results.append({
        "threshold": threshold,
        "precision": precision_score(y_test, y_pred_threshold),
        "recall": recall_score(y_test, y_pred_threshold),
        "f1_score": f1_score(y_test, y_pred_threshold),
        "false_positives": confusion_matrix(y_test, y_pred_threshold)[0, 1],
        "false_negatives": confusion_matrix(y_test, y_pred_threshold)[1, 0],
        "true_positives": confusion_matrix(y_test, y_pred_threshold)[1, 1]
    })

threshold_results_df = pd.DataFrame(threshold_results)
threshold_results_df

,threshold,precision,recall,f1_score,false_positives,false_negatives,true_positives
0,0.30,0.439490,0.922460,0.595341,440,29,345
1,0.35,0.459610,0.882353,0.604396,388,44,330
2,0.40,0.475483,0.855615,0.611270,353,54,320
3,0.45,0.494400,0.826203,0.618619,316,65,309
4,0.50,0.514680,0.796791,0.625393,281,76,298
5,0.55,0.539326,0.770053,0.634361,246,86,288
6,0.60,0.560254,0.708556,0.625738,208,109,265


## Threshold Analysis

After selecting XGBoost as the strongest model, different classification thresholds were tested to evaluate the tradeoff between recall, precision, false positives, and false negatives.

The goal is not only to maximize recall, but also to control unnecessary retention costs caused by false positives.

### Threshold Selection

I would recommend a threshold of 0.55 for the XGBoost model.

At this threshold, the model catches 288 churners and misses 86 churners. It also creates 246 false positives. Compared with the default threshold of 0.50, threshold 0.55 reduces false positives from 281 to 246, while recall decreases from 0.7968 to 0.7701.

I would not choose 0.30 because although it has the highest recall, it creates 440 false positives, which could lead to too many unnecessary retention offers. I would not choose 0.60 because it improves precision, but recall drops to 0.7086 and the model misses 109 churners.

Threshold 0.55 gives the best F1-score in the tested range and provides a more balanced tradeoff between catching churners and controlling retention costs.

### Final Model Recommendation

The selected model is XGBoost with a classification threshold of 0.55.

XGBoost was selected because it achieved the strongest overall performance among the tested models, with the best recall, F1-score, and ROC-AUC. Since this project prioritizes detecting customers likely to churn, recall is the most important metric.

The threshold was adjusted from 0.50 to 0.55 because it provided a better balance between recall and precision. At threshold 0.55, the model catches 288 out of 374 churners and misses 86. It also reduces false positives compared with the default threshold, which helps control unnecessary retention costs.

The main weakness is that precision remains moderate, meaning some customers flagged as churn risks would not actually churn. Therefore, the model should be used as a decision-support tool, not as an automatic discount system.

Business recommendation: use the model to prioritize customers for retention campaigns. Customers predicted as high churn risk should receive targeted retention actions, but expensive offers should be reserved for the highest-risk or highest-value customers.

Future improvements could include hyperparameter tuning for XGBoost, testing LightGBM or CatBoost, adding customer value segmentation, and evaluating retention strategies based on different intervention costs.

In [16]:
import joblib

# Save final selected model and threshold

final_model = xgb_model
final_threshold = 0.55

joblib.dump(final_model, "../models/xgboost_churn_model.pkl")
joblib.dump(final_threshold, "../models/xgboost_threshold.pkl")

print("Saved final XGBoost model and threshold.")

Saved final XGBoost model and threshold.


In [17]:
import os

model_files = [
    "../models/xgboost_churn_model.pkl",
    "../models/xgboost_threshold.pkl"
]

for file in model_files:
    print(file, "exists:", os.path.exists(file))

../models/xgboost_churn_model.pkl exists: True
../models/xgboost_threshold.pkl exists: True
